# Stage 2 preprocessing and split audit

This working notebook prepares data for a later comparison of custom PyTorch CNN and GRU models. It constructs neither model, loads no pretrained assets and performs no training, tuning or test scoring. Stage 1 files are read-only inputs.

Requirements: `CT118-3-3-ODL - Assignment marking scheme (1).doc`, Marking Scheme/Rubrics sections, assigns 20/20/30/10/10/10 marks to selection/preparation/building/tuning/evaluation/conclusion. Word page mapping is unavailable. `ODL_LECTURE#6_TRANSCRIPT.pdf` p.16 contains inconsistent preparation marks; the primary scheme governs. `ODL_LECTURE#13_TRANSCRIPT.pdf` pp.4-5 supports training-only vocabulary fitting and explains padding. Classroom vocabulary/cap examples are not prescribed settings. `ODL_LECTURE#1_TRANSCRIPT.pdf` pp.4-5 supports the postprocessing minimum and original models without pretrained weights; p.14 allows PyTorch. See the working requirements cross-check for remaining source limitations.

This is not a submission report or AI acknowledgement. Full assignment EDA, literature review and model work remain outside this notebook.

In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata
import numpy as np
import pandas as pd
import torch
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'src/stage2.py').exists():
    ROOT = ROOT.parent
assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from stage2 import run, tokenise, make_batch, preservation_manifest
before_stage1 = preservation_manifest()
versions = {p:importlib.metadata.version(p) for p in ['torch','pandas','numpy','regex','scikit-learn','nbformat','nbclient']}
print('Project Python:', sys.executable)
display(versions)

Project Python: <local-path-redacted>

{'torch': '2.14.0+cpu',
 'pandas': '3.0.5',
 'numpy': '2.5.3',
 'regex': '2026.9.10',
 'scikit-learn': '1.9.1',
 'nbformat': '5.11.1',
 'nbclient': '0.11.0'}

## Fixed text rules and duplicate policy

In order: Unicode NFC; collapse Unicode whitespace runs and trim ends; replace HTTP/HTTPS or www URLs with `<URL>`; replace mentions with `<USER>`; tokenise. The complete token list, encoded unambiguously as JSON, is the duplicate key. Case remains unchanged. No HTML unescaping, stopword removal, stemming or accent removal is used.

URLs stop at whitespace or angle brackets. Trailing `. , ! ? ; : ) ] }` are retained as prose punctuation; punctuation inside a URL is part of the replaced URL. This is a heuristic, and a URL genuinely ending with those characters can be interpreted differently. Mention recognition uses ASCII letters/digits/underscores after @, excluding email-like embedded matches. Different airline/user accounts therefore lose their identity.

The tokenizer keeps word/number/underscore runs, optional leading # and internal straight/curly apostrophes. Other non-whitespace content is kept as Unicode grapheme clusters, preserving emoji joiners and modifiers. Punctuation is retained as individual grapheme tokens, so punctuation-adjacent whitespace may cease to distinguish inputs. Negation and contractions remain, including `no`, `not`, `never`, `can't` and `don't`.

Exclude empty token lists and invalid labels, quarantine every row in a token group with disagreeing labels, then keep the first original source record in each agreeing group. The input already has unique IDs and Stage 1 text keys. One representative per sequence prevents repetition from adding weight to either training or evaluation. No confidence-based or majority relabelling. All excluded rows and source mappings are retained in Stage 2 outputs. Optional metadata is never an exclusion criterion. The minimum applies to distinct examples before splitting, not to the training partition alone.

In [2]:
assert tokenise("  I can't!! 😡\tnever?  ") == ['I', "can't", '!', '!', '😡', 'never', '?']
assert tokenise('👩‍✈️?!') == ['👩‍✈️', '?', '!']
assert tokenise('@United not happy https://example.com/a?!') == ['<USER>','not','happy','<URL>','?','!']
assert tokenise('Great!') != tokenise('great!')
assert tokenise('e\u0301') == ['é']
assert tokenise('@United Thanks!') == tokenise('@Delta Thanks!')
assert tokenise("don't #happy 😊") == ["don't",'#happy','😊']
print('Sentiment-cue and new-duplicate policy checks passed.')

Sentiment-cue and new-duplicate policy checks passed.


In [3]:
summary = run()
display(pd.DataFrame(summary['removal_ledger']))
display({k:summary[k] for k in ['stage1_rows','initial_unique_token_sequences','repeated_groups',
    'rows_in_repeated_groups','conflicting_groups','conflicting_rows','distinct_rows_before_split',
    'distinct_token_sequences_before_split','minimum_10000_passed']})
assert summary['minimum_10000_passed']
print('The minimum passed; the existing split was reused without changes.')

,step,before,removed,remaining
0,empty_token_sequence,14398,0,14398
1,invalid_label,14398,0,14398
2,quarantine_conflicting_token_group,14398,48,14350
3,duplicate_token_sequence_keep_first_source_row,14350,79,14271


{'stage1_rows': 14398,
 'initial_unique_token_sequences': 14285,
 'repeated_groups': 63,
 'rows_in_repeated_groups': 176,
 'conflicting_groups': 14,
 'conflicting_rows': 48,
 'distinct_rows_before_split': 14271,
 'distinct_token_sequences_before_split': 14271,
 'minimum_10000_passed': True}

The minimum passed; the existing split was reused without changes.


## Shared split and training-only fitting

The existing seed-42 stratified 70/15/15 assignments are frozen and reused, not regenerated. Hashes in `config/stage2_frozen_inputs.json` pin the split CSVs, manifest, representatives, removal ledger and training-length table. Group identities and source IDs are saved with assignments. There is one shared manifest for both future models. Initial grouping uses complete token sequences before any vocabulary or length fitting. All representative groups have one row; no group can be split apart.

Vocabulary counts use training only. Retain every training-observed token (minimum frequency one), including all observed negation and singleton words. Order by descending training frequency then token text for reproducible IDs. Reserve `<PAD>=0`, `<UNK>=1`, `<URL>=2`, `<USER>=3`. Validation/test-only words do not enter the vocabulary. Literal special-marker spellings in input are a tokenizer limitation to consider during review. `<UNK>` is a lost word identity; `<PAD>` is absent text.

Only predefined structural checks access held-out data: encoding, source/key overlaps, ID ranges and padding consistency. There is no inspection of test predictions, examples, OOV rates or length distributions to select settings. Stratification necessarily uses labels for partition construction. The test file is held for final evaluation, and validation is reserved for tuning. Optional loss weights are computed from training counts using N/(3 × class count); no loss is constructed or applied.

In [4]:
display(pd.DataFrame(summary['splits']).T)
display(pd.read_csv(ROOT / 'results/stage2/split_class_distribution.csv'))
config = json.loads((ROOT / 'data/processed/stage2/config.json').read_text(encoding='utf-8'))
print('Vocabulary size including reserved tokens:', config['vocabulary_size'])
display(config['special_ids'])
display(config['optional_training_class_weights'])

,rows,distinct_sequences
train,9989,9989
validation,2141,2141
test,2141,2141


,split,sentiment,rows
0,train,negative,6348
1,train,neutral,2079
2,train,positive,1562
3,validation,negative,1360
4,validation,neutral,446
5,validation,positive,335
6,test,negative,1361
7,test,neutral,446
8,test,positive,334


Vocabulary size including reserved tokens: 13898


{'<PAD>': 0, '<UNK>': 1, '<URL>': 2, '<USER>': 3}

{'negative': 0.5245221592102499,
 'neutral': 1.6015712682379348,
 'positive': 2.1316688006828852}

## Training-only sequence length evidence

Compare the median, p90, p95, p99 and maximum, using the higher observed quantile. For each cap, count affected training tweets, discarded training tokens and fixed-length padding. The selection rule was set before seeing these measurements: choose the smallest listed cap that truncates no more than 1% of training tweets and 0.1% of training tokens; otherwise use the maximum. These tolerances are design choices, not assignment requirements. They prioritise preserving short-tweet sentiment cues.

The existing cap of 46 is frozen; recomputation verifies the original training-only evidence. The common cap applies unchanged to both models and all partitions. Longer inputs keep their first cap tokens; this can discard ending sentiment or negation, a limitation even if training has no truncation. Arrays are stored right-padded at the cap for portability; the batch helper removes unnecessary padding down to the longest retained sequence in the batch. Fixed-padding figures below are therefore upper-bound storage/computation comparisons, not measured model runtimes.

In [5]:
length_table = pd.read_csv(ROOT / 'results/stage2/training_length_comparison.csv')
display(length_table)
print('Chosen candidate:', config['length_candidate_selected'])
print('Chosen cap:', config['length_cap'])
max_slots = int(length_table.loc[length_table.candidate.eq('maximum'),'fixed_batch_token_slots'].iloc[0])
p99_slots = int(length_table.loc[length_table.candidate.eq('p99'),'fixed_batch_token_slots'].iloc[0])
print('Fixed-size token-slot saving for p99 versus maximum (%):',round(100*(1-p99_slots/max_slots),2))

,candidate,length,truncated_tweets,truncated_tweets_percent,discarded_tokens,discarded_tokens_percent,fixed_padding_tokens,fixed_padding_percent,fixed_batch_token_slots
0,p50,22,4766,47.712484,26731,12.875336,38875,17.689913,219758
1,p90,30,880,8.809691,2500,1.204158,94556,31.553375,299670
2,p95,32,382,3.824207,1024,0.493223,113058,35.369531,319648
3,p99,35,96,0.961057,252,0.121379,142253,40.688472,349615
4,maximum,46,0,0.000000,0,0.000000,251880,54.816820,459494


Chosen candidate: maximum
Chosen cap: 46
Fixed-size token-slot saving for p99 versus maximum (%): 23.91


## Padding safeguards and leakage gate

The model-free batch helper supplies integer inputs, true lengths and a Boolean real-token mask. Future GRU code must use those lengths (for example packed sequences); future CNN code must mask padded positions appropriately before pooling, accounting for its kernel and boundary choices. A zero padding embedding alone does not prove that padding has no model effect. Those model-level safeguards cannot be verified until model implementation, which is not performed here.

Full-token uniqueness can be lost after genuinely unseen words map to `<UNK>` or inputs are truncated. Check identical encoded sequences within and across splits. Encoded collisions are non-fatal representation diagnostics, not automatically evidence of leakage. Full-token/source overlaps and invalid fitting or padding remain hard failures. Do not remove further records, grow vocabulary using held-out words, change caps using held-out evidence or reshuffle partitions to hide failures. Preserve this run for review.

In [6]:
example = make_batch(np.array([[4,5,0,0],[6,0,0,0]],dtype=np.int64),[2,1])
assert example['input_ids'].shape == (2,2)
assert example['attention_mask'].tolist() == [[True,True],[True,False]]
assert example['lengths'].tolist() == [2,1]
checks = json.loads((ROOT / 'results/stage2/checks.json').read_text(encoding='utf-8'))
display(pd.Series(checks,name='passed').to_frame())
print('Encoded collision rows:', summary['encoded_collision_rows'])
print('Cross-split encoded collision groups:', summary['cross_split_encoded_collision_groups'])
print('READINESS:', summary['status'])
assert before_stage1 == preservation_manifest()
print('Stage 1 inputs and outputs remain byte-identical.')

,passed
disjoint_source_ids,True
vocabulary_training_only,True
all_required_training_tokens_in_vocab,True
full_sequence_uniqueness,True
train_padding_valid,True
train_ids_in_range,True
no_training_unknown_tokens,True
validation_padding_valid,True
validation_ids_in_range,True
test_padding_valid,True


Encoded collision rows: 5
Cross-split encoded collision groups: 1
READINESS: complete
Stage 1 inputs and outputs remain byte-identical.


## Review outcome and saved files

The `READINESS` value above is authoritative. A notebook completing its cells is not proof that the leakage gate passed. If it says STOP, an integrity check failed and needs review. Encoded collisions alone do not cause STOP. No held-out rows are removed because they collide. The proposed training-only UNK exposure policy and architectures are in `docs/stage3_proposal.md`; neither is implemented here.

`data/processed/stage2/`: distinct rows, train/validation/test CSVs, shared split manifest, training-only vocabulary/config and encoded NPZ arrays. `results/stage2/`: every exclusion, repeated/conflicting groups, source outcomes, length comparisons, class bookkeeping, collision IDs, checks and Stage 1 preservation hashes. `results/stage2/notebook_execution.json` records execution and repeat-output hashes. Read IDs as strings. Exact preprocessing does not address semantic near-duplicates or establish annotation correctness. The earlier 14,250 result remains unexplained and is not a target.

Assistance: Codex generated the preprocessing, notebook and verification code, read assignment materials, and executed checks. No model or pretrained weights were created/loaded, no fitting of models occurred, and no assignment report or AI acknowledgement was written. See the separate working assistance record for the actual scope.